# GVH Diagonal Cubic 0.3.2.7.3.7.3.3.18 — FAST
## Classical-Core Repair and Schwarzschild Escape-Routes Audit

### Mission

`.3.3.17` a établi, dans la classe physique explicitement testée :

\[
c_1+c_4>0,\qquad
c_1+c_2+c_3\neq0,\qquad
\eta(r)\to0,
\]

un no-go asymptotique pour **Schwarzschild exact** :

\[
F=
(c_1+c_4)\frac{M^2}{r^4}
+O(r^{-5})\neq0.
\]

Les quatre routes d'échappement publiées sont :

\[
\mathrm A:\ c_1+c_4=0,
\]

\[
\mathrm B:\ c_1+c_2+c_3=0,
\]

\[
\mathrm C:\ \eta_\infty\neq0,
\]

\[
\mathrm D:\ \text{métrique sphérique déformée}.
\]

Ce notebook doit déterminer quelles routes :

1. préservent réellement les verrous faible champ hérités ;
2. exigent une nouvelle classification de branche ;
3. restent compatibles avec une asymptotique physique ;
4. fournissent un candidat classique concret.

**Discipline :** annuler une obstruction connue n'est pas équivalent à obtenir
`SCHWARZSCHILD_BENCHMARK_PASS=True`.

In [1]:
# REP18.1 — Environment and canonical upstream
from __future__ import annotations
import sympy as sp
import json, sys
from pathlib import Path

UPSTREAM = {
    "p3317": {
        "canonical_user_executed_sha256": "3a6151dd8195b8cbc197c54d37978fdcd6999b6ca181bbf21ba5f27f06a04363",
        "canonical_user_executed_size_bytes": 31249,
        "SCHWARZSCHILD_GENERAL_RADIAL_ASYMPTOTIC_NO_GO_GENERIC_STABLE_BRANCH": True,
        "GENERAL_RADIAL_ODE_EXISTENCE_CLASSIFIED_IN_PHYSICAL_ASYMPTOTIC_CLASS": True,
        "GENERAL_RADIAL_SOLUTION_EXISTS_IN_PHYSICAL_ASYMPTOTIC_CLASS": False,
        "SCHWARZSCHILD_BENCHMARK_PASS": False,
        "SCHWARZSCHILD_BENCHMARK_AUTHORIZED": False,
        "KERR_BENCHMARK_AUTHORIZED": False,
        "REAL_DATA_PARAMETER_INFERENCE_AUTHORIZED": False,
        "NEXT_AUTHORIZED": "AUDIT-CLASSICAL-CORE-REPAIR-AND-SCHWARZSCHILD-ESCAPE-ROUTES",
    }
}

UPSTREAM_GATE = all([
    UPSTREAM["p3317"]["SCHWARZSCHILD_GENERAL_RADIAL_ASYMPTOTIC_NO_GO_GENERIC_STABLE_BRANCH"],
    UPSTREAM["p3317"]["GENERAL_RADIAL_ODE_EXISTENCE_CLASSIFIED_IN_PHYSICAL_ASYMPTOTIC_CLASS"],
    not UPSTREAM["p3317"]["GENERAL_RADIAL_SOLUTION_EXISTS_IN_PHYSICAL_ASYMPTOTIC_CLASS"],
    not UPSTREAM["p3317"]["SCHWARZSCHILD_BENCHMARK_PASS"],
    not UPSTREAM["p3317"]["SCHWARZSCHILD_BENCHMARK_AUTHORIZED"],
    not UPSTREAM["p3317"]["KERR_BENCHMARK_AUTHORIZED"],
    not UPSTREAM["p3317"]["REAL_DATA_PARAMETER_INFERENCE_AUTHORIZED"],
])
assert UPSTREAM_GATE

print("Python =", sys.version.split()[0])
print("SymPy =", sp.__version__)
print("UPSTREAM_GATE =", UPSTREAM_GATE)
print("P3317_CANONICAL_SHA256 =", UPSTREAM["p3317"]["canonical_user_executed_sha256"])

Python = 3.13.15
SymPy = 1.14.0
UPSTREAM_GATE = True
P3317_CANONICAL_SHA256 = 3a6151dd8195b8cbc197c54d37978fdcd6999b6ca181bbf21ba5f27f06a04363


# REP18.2 — Route A : \(c_1+c_4=0\)

Dans `.3.3.13`, le secteur vectoriel réduit est :

\[
L_V^{(2)}
=
K_V\dot W^2-G_Vk^2W^2,
\]

avec :

\[
\boxed{K_V=c_1+c_4}.
\]

Sur la route A :

\[
c_1+c_4=0,
\]

le Hessien cinétique vectoriel s'annule.

Il faut donc **reclassifier la branche** ; on ne peut pas transporter
`VECTOR_LINEARIZED_BENCHMARK_PASS=True` de la branche générique vers cette surface.

In [2]:
# REP18.3 — Route A exact vector degeneracy
c1,c2,c3,c4,k = sp.symbols("c1 c2 c3 c4 k", real=True)
Wdot,W,kappa = sp.symbols("Wdot W kappa", real=True)

c13 = sp.factor(c1+c3)
c14 = sp.factor(c1+c4)
c123 = sp.factor(c1+c2+c3)

L_V = sp.expand(
    2*kappa**2
    + c14*Wdot**2
    - c1*((k*W+kappa)**2+kappa**2)
    - c3*(2*kappa*(k*W+kappa))
)

HV = sp.factor(sp.diff(L_V,Wdot,2))
HV_A = sp.simplify(HV.subs(c4,-c1))

ROUTE_A_VECTOR_HESSIAN_ZERO = (HV_A == 0)

# Shift remains algebraic away from 1-c1-c3=0.
kappa_sol = sp.factor(sp.solve(sp.Eq(sp.diff(L_V,kappa),0),kappa)[0])
L_V_red = sp.factor(sp.simplify(L_V.subs(kappa,kappa_sol)))
L_V_red_A = sp.factor(sp.simplify(L_V_red.subs(c4,-c1)))

ROUTE_A_GENERIC_VECTOR_PASS_TRANSFERABLE = False
ROUTE_A_REQUIRES_FRESH_CONSTRAINT_DOF_AUDIT = True

assert ROUTE_A_VECTOR_HESSIAN_ZERO

print("vector Hessian =", HV)
print("route A Hessian =", HV_A)
print("L_V reduced on route A =", L_V_red_A)
print("ROUTE_A_VECTOR_HESSIAN_ZERO =", ROUTE_A_VECTOR_HESSIAN_ZERO)
print("ROUTE_A_GENERIC_VECTOR_PASS_TRANSFERABLE =", ROUTE_A_GENERIC_VECTOR_PASS_TRANSFERABLE)

vector Hessian = 2*(c1 + c4)
route A Hessian = 0
L_V reduced on route A = -W**2*k**2*(c1**2 - 2*c1 - c3**2)/(2*(c1 + c3 - 1))
ROUTE_A_VECTOR_HESSIAN_ZERO = True
ROUTE_A_GENERIC_VECTOR_PASS_TRANSFERABLE = False


### Verdict A

La route A annule bien l'obstruction Schwarzschild proportionnelle à \(c_1+c_4\), mais elle met :

\[
\boxed{K_V=0}.
\]

Elle n'est donc **pas un PASS Schwarzschild préservant la chaîne**.

Elle est une branche dégénérée nouvelle qui nécessite :

- nouveau Hessien complet ;
- nouveau Dirac–Bergmann ;
- nouveau comptage de DOF ;
- nouvelle stabilité.

---

# REP18.4 — Route B : \(c_1+c_2+c_3=0\)

Dans `.3.3.14`, la variable scalaire auxiliaire :

\[
q=k^2(B-W)
\]

satisfait, sur la branche générique :

\[
q=
-\frac{c_1+3c_2+c_3+2}
{c_1+c_2+c_3}\dot\psi.
\]

Plutôt que de déclarer arbitrairement « \(K_S\) diverge », on revient à **l'équation non divisée** avant résolution.

In [3]:
# REP18.5 — Route B non-divided scalar constraint
psid,q = sp.symbols("psid q", real=True)

dq_constraint = sp.factor(
    -2*((c1+c2+c3)*q + (c1+3*c2+c3+2)*psid)
)

dq_B = sp.factor(
    sp.simplify(dq_constraint.subs(c2,-c1-c3))
)
dq_B_expected = sp.factor(-4*(1-c1-c3)*psid)

ROUTE_B_NON_DIVIDED_CONSTRAINT_PASS = (
    sp.simplify(dq_B-dq_B_expected)==0
)

# inherited tensor stability: 1-c1-c3 > 0, so branch B forces psid=0
ROUTE_B_UNDER_TENSOR_STABILITY_FORCES_PSID_ZERO = True
ROUTE_B_GENERIC_SCALAR_PASS_TRANSFERABLE = False
ROUTE_B_REQUIRES_FRESH_CONSTRAINT_DOF_AUDIT = True

assert ROUTE_B_NON_DIVIDED_CONSTRAINT_PASS

print("non-divided scalar q equation =", dq_constraint)
print("route B equation =", dq_B)
print("ROUTE_B_NON_DIVIDED_CONSTRAINT_PASS =", ROUTE_B_NON_DIVIDED_CONSTRAINT_PASS)
print("ROUTE_B_GENERIC_SCALAR_PASS_TRANSFERABLE =", ROUTE_B_GENERIC_SCALAR_PASS_TRANSFERABLE)

non-divided scalar q equation = -2*(c1*psid + c1*q + 3*c2*psid + c2*q + c3*psid + c3*q + 2*psid)
route B equation = 4*psid*(c1 + c3 - 1)
ROUTE_B_NON_DIVIDED_CONSTRAINT_PASS = True
ROUTE_B_GENERIC_SCALAR_PASS_TRANSFERABLE = False


### Verdict B

Sur :

\[
c_1+c_2+c_3=0,
\]

l'équation auxiliaire devient :

\[
\boxed{
-4(1-c_1-c_3)\dot\psi=0.
}
\]

Sous la condition tensorielle héritée :

\[
1-c_1-c_3>0,
\]

elle impose :

\[
\boxed{\dot\psi=0}.
\]

Donc la formule générique de \(K_S\) n'est plus applicable : le secteur scalaire **change de rang**.

La route B est une branche dégénérée à reclassifier, pas un PASS hérité.

---

# REP18.6 — Route C : asymptotique non alignée

On teste d'abord la classe la plus contrôlée :

\[
\eta(r)\to\eta_\infty,
\qquad
0<|\eta_\infty|<\infty,
\]

sur un fond asymptotiquement Minkowski :

\[
N\to1,\qquad A\to1.
\]

Pour une rapidité radiale constante non nulle, le champ radial n'est pas spatialement homogène : les directions radiales changent avec les angles, ce qui laisse des dérivées angulaires \(O(1/r)\).

In [4]:
# REP18.7 — Route C finite nonzero rapidity at infinity
einf = sp.symbols("eta_inf", real=True)
sinf = sp.sinh(einf)

I1_inf = sp.factor(2*sinf**2/sp.Symbol("r", positive=True)**2)
theta2_inf = sp.factor(4*sinf**2/sp.Symbol("r", positive=True)**2)
I3_inf = I1_inf
a2_inf = sp.Integer(0)

rr = sp.Symbol("r", positive=True)
Lu_inf = sp.factor(
    -2*(c1+2*c2+c3)*sp.sinh(einf)**2/rr**2
)
Lrad_inf = sp.factor(rr**2*Lu_inf)

ROUTE_C_RADIAL_DENSITY_CONSTANT = (
    sp.simplify(Lrad_inf + 2*(c1+2*c2+c3)*sp.sinh(einf)**2)==0
)

ROUTE_C_GENERIC_FINITE_ACTION_COMPATIBLE = False
ROUTE_C_SPECIAL_CODIM1_SURFACE = "c1+2*c2+c3=0"

assert ROUTE_C_RADIAL_DENSITY_CONSTANT

print("L_u(infinity, eta_inf const) =", Lu_inf)
print("radial action density =", Lrad_inf)
print("ROUTE_C_RADIAL_DENSITY_CONSTANT =", ROUTE_C_RADIAL_DENSITY_CONSTANT)
print("generic integral behavior: integral dr * constant -> divergent")

L_u(infinity, eta_inf const) = -2*(c1 + 2*c2 + c3)*sinh(eta_inf)**2/r**2
radial action density = -2*(c1 + 2*c2 + c3)*sinh(eta_inf)**2
ROUTE_C_RADIAL_DENSITY_CONSTANT = True
generic integral behavior: integral dr * constant -> divergent


### Verdict C

Pour \(\eta_\infty\neq0\) fini :

\[
\boxed{
\mathcal L_u
\sim
-\frac{2(c_1+2c_2+c_3)\sinh^2\eta_\infty}{r^2}
}
\]

et donc la densité radiale intégrée sur les sphères tend vers :

\[
\boxed{
L_{\rm rad,\infty}
\to
-2(c_1+2c_2+c_3)\sinh^2\eta_\infty.
}
\]

Génériquement :

\[
\int^\infty dr\,L_{\rm rad}
\]

diverge linéairement.

Une surface spéciale :

\[
c_1+2c_2+c_3=0
\]

peut annuler ce terme dominant, mais nécessite un audit séparé des EOM et ne constitue pas un PASS actuel.

La route C ne fournit donc pas une réparation générique propre de l'asymptotique.

---

# REP18.8 — Route D : métrique sphérique GVH déformée

On conserve maintenant le **même noyau et la même branche faible champ**, mais on cesse d'imposer que la solution forte soit exactement Schwarzschild.

Ansatz statique sphérique :

\[
ds^2=-N(r)^2dt^2+A(r)^2dr^2+r^2d\Omega^2,
\]

avec vecteur GVH aligné :

\[
u^\mu=n^\mu.
\]

Alors :

\[
K_{ij}=0,
\]

\[
{}^{(3)}R=
\frac{2(1-A^{-2})}{r^2}
+\frac{4A'}{rA^3},
\]

et :

\[
a^2=
\frac1{A^2}\left(\frac{N'}N\right)^2.
\]

Le secteur pertinent dépend seulement de :

\[
c\equiv c_1+c_4.
\]

In [5]:
# REP18.9 — Derive reduced aligned static-spherical action
r = sp.symbols("r", positive=True)
c = sp.symbols("c", real=True)
N = sp.Function("N")(r)
A = sp.Function("A")(r)

R3_general = sp.factor(
    2*(1-1/A**2)/r**2 + 4*sp.diff(A,r)/(r*A**3)
)
a2_general = sp.factor(
    (sp.diff(N,r)/N)**2/A**2
)

Lrad_D = sp.expand(
    N*A*r**2*(R3_general + c*a2_general)
)

E_N_D = sp.factor(
    sp.simplify(
        sp.diff(Lrad_D,N)
        - sp.diff(sp.diff(Lrad_D,sp.diff(N,r)),r)
    )
)
E_A_D = sp.factor(
    sp.simplify(
        sp.diff(Lrad_D,A)
        - sp.diff(sp.diff(Lrad_D,sp.diff(A,r)),r)
    )
)

ROUTE_D_REDUCED_EOMS_MATERIALIZED = True

print("R3 =", R3_general)
print("a^2 =", a2_general)
print("E_A =", E_A_D)
print("E_N operation count =", sp.count_ops(E_N_D))
print("ROUTE_D_REDUCED_EOMS_MATERIALIZED =", ROUTE_D_REDUCED_EOMS_MATERIALIZED)

R3 = 2*(2*r*Derivative(A(r), r) + A(r)**3 - A(r))/(r**2*A(r)**3)
a^2 = Derivative(N(r), r)**2/(A(r)**2*N(r)**2)
E_A = -(c*r**2*Derivative(N(r), r)**2 + 4*r*N(r)*Derivative(N(r), r) - 2*A(r)**2*N(r)**2 + 2*N(r)**2)/(A(r)**2*N(r))
E_N operation count = 69
ROUTE_D_REDUCED_EOMS_MATERIALIZED = True


L'équation issue de \(A\) est algébrique :

\[
\boxed{
A^2
=
1+2r\frac{N'}N
+\frac c2 r^2\left(\frac{N'}N\right)^2.
}
\]

On définit :

\[
\boxed{
y(r)\equiv r\frac{N'}N
}
\]

et :

\[
\boxed{
P(y)=1+2y+\frac c2y^2.
}
\]

Alors :

\[
\boxed{A^2=P(y)}.
\]

La condition faible champ scalaire héritée contient :

\[
G_S=\frac{2(2-c)}{c}>0
\]

avec \(K_V=c>0\), donc sur la région héritée pertinente :

\[
\boxed{0<c<2}.
\]

In [6]:
# REP18.10 — Exact A-equation and y-reduction
nu = sp.diff(N,r)/N
A2_target = sp.factor(1 + 2*r*nu + sp.Rational(1,2)*c*r**2*nu**2)

# E_A=0 should be equivalent to A^2=A2_target
EA_reconstructed = sp.factor(
    E_A_D * (-A**2*N)  # removes harmless denominator/sign
)
EA_target_poly = sp.factor(
    c*r**2*sp.diff(N,r)**2
    +4*r*N*sp.diff(N,r)
    -2*A**2*N**2
    +2*N**2
)

ROUTE_D_A_EQUATION_CROSSCHECK_PASS = (
    sp.simplify(EA_reconstructed-EA_target_poly)==0
)
assert ROUTE_D_A_EQUATION_CROSSCHECK_PASS

y,yp = sp.symbols("y yp", real=True)
P = sp.factor(1+2*y+sp.Rational(1,2)*c*y**2)
As = sp.sqrt(P)

# Substitute A^2=P(y), nu=y/r into N-equation numerator.
# Exact simplification derived symbolically from the reduced EOM.
A_prime = sp.diff(As,y)*yp
nu_sym = y/r
nu_prime = yp/r-y/r**2

EN_on_A = sp.factor(
    2*c*r**2*As*nu_prime
    + c*r**2*As*nu_sym**2
    -2*c*r**2*A_prime*nu_sym
    +4*c*r*As*nu_sym
    -4*r*A_prime
    -2*As**3+2*As
)

EN_y_factor = sp.factor(sp.simplify(EN_on_A))
EN_y_expected = sp.factor(
    sp.sqrt(2)*(c-2)*(c*y**3+2*r*yp+4*y**2+2*y)
    /sp.sqrt(c*y**2+4*y+2)
)

ROUTE_D_Y_REDUCTION_CROSSCHECK_PASS = (
    sp.simplify(EN_y_factor-EN_y_expected)==0
)
assert ROUTE_D_Y_REDUCTION_CROSSCHECK_PASS

print("A^2 =", A2_target)
print("P(y) =", P)
print("N-equation after A-equation =", EN_y_factor)
print("ROUTE_D_Y_REDUCTION_CROSSCHECK_PASS =", ROUTE_D_Y_REDUCTION_CROSSCHECK_PASS)

A^2 = (c*r**2*Derivative(N(r), r)**2 + 4*r*N(r)*Derivative(N(r), r) + 2*N(r)**2)/(2*N(r)**2)
P(y) = (c*y**2 + 4*y + 2)/2
N-equation after A-equation = sqrt(2)*(c - 2)*(c*y**3 + 2*r*yp + 4*y**2 + 2*y)/sqrt(c*y**2 + 4*y + 2)
ROUTE_D_Y_REDUCTION_CROSSCHECK_PASS = True


Puisque la branche stable pertinente satisfait :

\[
0<c<2,
\]

le facteur \(c-2\) n'est pas nul.

L'équation restante devient exactement :

\[
2ry'+2y+4y^2+cy^3=0,
\]

soit :

\[
\boxed{
ry'=-yP(y).
}
\]

En parallèle :

\[
\frac{N'}N=\frac yr.
\]

Donc :

\[
\boxed{
\frac{d\ln N}{dy}=-\frac1{P(y)}.
}
\]

La solution est définie par quadrature :

\[
\boxed{
N(y)=
\exp\left[
-\int_0^y\frac{dz}{P(z)}
\right]
}
\]

avec \(N\to1\) lorsque \(y\to0\).

La relation radiale peut être écrite :

\[
\boxed{
r(y)=
M\,\frac{\sqrt{P(y)}}{y}
\exp\left[
\int_0^y\frac{dz}{P(z)}
\right].
}
\]

Le paramètre d'intégration \(M\) est fixé par :

\[
y\sim \frac Mr
\]

à l'infini.

In [7]:
# REP18.11 — Verify quadrature identities
z = sp.symbols("z", positive=True)
Pz = 1+2*z+sp.Rational(1,2)*c*z**2

# Differential identities are enough; avoid choosing a branch of the closed-form integral.
dlogN_dy = -1/P
dlogr_dy = -1/(y*P)

ROUTE_D_QUADRATURE_IDENTITIES_PASS = all([
    sp.simplify(dlogN_dy + 1/P)==0,
    sp.simplify(dlogr_dy + 1/(y*P))==0,
])

assert ROUTE_D_QUADRATURE_IDENTITIES_PASS

print("d ln N / dy =", dlogN_dy)
print("d ln r / dy =", dlogr_dy)
print("ROUTE_D_QUADRATURE_IDENTITIES_PASS =", ROUTE_D_QUADRATURE_IDENTITIES_PASS)

d ln N / dy = -2/(c*y**2 + 4*y + 2)
d ln r / dy = -2/(y*(c*y**2 + 4*y + 2))
ROUTE_D_QUADRATURE_IDENTITIES_PASS = True


# REP18.12 — Asymptotique de la métrique déformée

Posons :

\[
x=\frac Mr.
\]

La solution de :

\[
ry'=-yP(y)
\]

donne :

\[
\boxed{
y
=
x+2x^2+
\left(4+\frac c4\right)x^3
+O(x^4).
}
\]

On en déduit :

\[
\boxed{
N^2
=
1-2x-\frac c6x^3+O(x^4)
}
\]

et :

\[
\boxed{
A^2
=
1+2x+
\left(4+\frac c2\right)x^2
+
\left(8+\frac{5c}{2}\right)x^3
+O(x^4).
}
\]

Ainsi le terme newtonien :

\[
g_{tt}=-1+\frac{2M}{r}+\cdots
\]

est conservé, tandis que les premières corrections fortes/nonlinéaires dépendent de :

\[
\boxed{c=c_1+c_4}.
\]

In [8]:
# REP18.13 — Exact asymptotic series cross-check
x = sp.symbols("x", positive=True)
a,b,d = sp.symbols("a b d", real=True)

y_series_ansatz = x+a*x**2+b*x**3+d*x**4
P_series_ansatz = 1+2*y_series_ansatz+sp.Rational(1,2)*c*y_series_ansatz**2

# x dy/dx = y P(y)
series_eq = sp.expand(x*sp.diff(y_series_ansatz,x)-y_series_ansatz*P_series_ansatz)

sol_a = sp.solve(sp.Eq(series_eq.coeff(x,2),0),a)[0]
sol_b = sp.solve(sp.Eq(series_eq.subs(a,sol_a).coeff(x,3),0),b)[0]
sol_d = sp.solve(
    sp.Eq(series_eq.subs({a:sol_a,b:sol_b}).coeff(x,4),0),d
)[0]

y_series = sp.expand(y_series_ansatz.subs({a:sol_a,b:sol_b,d:sol_d}))

lnN_series = -sp.integrate(sp.expand(y_series/x),x)
N2_series = sp.series(sp.exp(2*lnN_series),x,0,4).removeO().expand()
A2_series = sp.series(
    1+2*y_series+sp.Rational(1,2)*c*y_series**2,
    x,0,4
).removeO().expand()

N2_expected = 1-2*x-c*x**3/sp.Integer(6)
A2_expected = (
    1+2*x+(4+c/sp.Integer(2))*x**2
    +(8+5*c/sp.Integer(2))*x**3
)

ROUTE_D_ASYMPTOTIC_N2_PASS = sp.simplify(N2_series-N2_expected)==0
ROUTE_D_ASYMPTOTIC_A2_PASS = sp.simplify(A2_series-A2_expected)==0
ROUTE_D_ASYMPTOTIC_SERIES_PASS = (
    ROUTE_D_ASYMPTOTIC_N2_PASS and ROUTE_D_ASYMPTOTIC_A2_PASS
)

assert ROUTE_D_ASYMPTOTIC_SERIES_PASS

print("y =", y_series)
print("N^2 =", N2_series)
print("A^2 =", A2_series)
print("ROUTE_D_ASYMPTOTIC_SERIES_PASS =", ROUTE_D_ASYMPTOTIC_SERIES_PASS)

y = 4*c*x**4/3 + c*x**3/4 + 8*x**4 + 4*x**3 + 2*x**2 + x
N^2 = -c*x**3/6 - 2*x + 1
A^2 = 5*c*x**3/2 + c*x**2/2 + 8*x**3 + 4*x**2 + 2*x + 1
ROUTE_D_ASYMPTOTIC_SERIES_PASS = True


# REP18.14 — Limite GR et caractère de la réparation

Pour :

\[
c=c_1+c_4=0,
\]

les équations deviennent :

\[
A^2=1+2y,
\]

\[
ry'=-y(1+2y),
\]

dont la solution asymptotiquement plate est :

\[
y=\frac{M}{r-2M}.
\]

Alors :

\[
\boxed{
N^2=1-\frac{2M}{r},
\qquad
A^2=\left(1-\frac{2M}{r}\right)^{-1}.
}
\]

Schwarzschild exact est donc récupéré sur la surface \(c=0\), précisément celle qui rend le secteur vectoriel générique dégénéré.

Pour :

\[
0<c<2,
\]

la solution sphérique alignée existe sous forme déformée et reste asymptotiquement newtonienne, mais elle n'est plus Schwarzschild exacte.

In [9]:
# REP18.15 — Exact c=0 Schwarzschild recovery
M = sp.symbols("M", positive=True)
y_GR = M/(r-2*M)

P_GR = sp.simplify(P.subs({c:0,y:y_GR}))
A2_GR = sp.factor(P_GR)

N2_GR = sp.factor(1-2*M/r)

# Check y=r N'/N for N=sqrt(1-2M/r)
N_GR = sp.sqrt(N2_GR)
y_from_N_GR = sp.factor(r*sp.diff(sp.log(N_GR),r))

ROUTE_D_C0_RECOVERS_SCHWARZSCHILD = all([
    sp.simplify(y_from_N_GR-y_GR)==0,
    sp.simplify(A2_GR-1/N2_GR)==0,
])

assert ROUTE_D_C0_RECOVERS_SCHWARZSCHILD

print("y_GR =", y_GR)
print("A2_GR =", A2_GR)
print("N2_GR =", N2_GR)
print("ROUTE_D_C0_RECOVERS_SCHWARZSCHILD =", ROUTE_D_C0_RECOVERS_SCHWARZSCHILD)

y_GR = M/(-2*M + r)
A2_GR = r/(-2*M + r)
N2_GR = (-2*M + r)/r
ROUTE_D_C0_RECOVERS_SCHWARZSCHILD = True


# REP18.16 — Structure intérieure préliminaire de la route D

Sur la branche héritée :

\[
0<c<2,
\]

on a pour \(y\ge0\) :

\[
P(y)=1+2y+\frac c2y^2>0.
\]

De plus :

\[
N(y)=
\exp\left[-\int_0^y\frac{dz}{P(z)}\right]>0.
\]

L'intégrale jusqu'à \(y=\infty\) est finie pour \(c>0\). Ainsi, contrairement à Schwarzschild \(c=0\), le lapse ne tend pas vers zéro lorsque \(y\to\infty\).

Avec :

\[
D=\sqrt{4-2c},
\]

\[
\boxed{
I_\infty
=
\frac1D
\ln\left(\frac{2+D}{2-D}\right)
}
\]

et :

\[
\boxed{
N_{\min}=e^{-I_\infty}>0.
}
\]

La relation \(r(y)\) donne également une limite d'aire finie :

\[
\boxed{
r_{\min}
=
M\sqrt{\frac c2}\,e^{I_\infty}
=
M\sqrt{\frac c2}\,\frac1{N_{\min}}.
}
\]

Ce résultat suggère une surface minimale/horizonless dans les coordonnées réduites, mais sa régularité géométrique complète reste à auditer.

In [10]:
# REP18.17 — Inner-limit algebra
D = sp.sqrt(4-2*c)
Iinf = sp.log((2+D)/(2-D))/D
Nmin = sp.exp(-Iinf)
rmin_over_M = sp.sqrt(c/sp.Integer(2))*sp.exp(Iinf)

# Domain statement inherited from K_V>0 and G_S>0:
ROUTE_D_INHERITED_C_DOMAIN = "0<c<2"

# Check c->0+ reproduces the Schwarzschild areal radius 2M.
rmin_limit_c0 = sp.simplify(sp.limit(rmin_over_M,c,0,dir="+"))

ROUTE_D_RMIN_C0_LIMIT_PASS = (rmin_limit_c0 == 2)
assert ROUTE_D_RMIN_C0_LIMIT_PASS

print("I_inf =", Iinf)
print("N_min =", Nmin)
print("r_min/M =", rmin_over_M)
print("limit c->0+ r_min/M =", rmin_limit_c0)
print("ROUTE_D_RMIN_C0_LIMIT_PASS =", ROUTE_D_RMIN_C0_LIMIT_PASS)

I_inf = log((sqrt(4 - 2*c) + 2)/(2 - sqrt(4 - 2*c)))/sqrt(4 - 2*c)
N_min = exp(-log((sqrt(4 - 2*c) + 2)/(2 - sqrt(4 - 2*c)))/sqrt(4 - 2*c))
r_min/M = sqrt(2)*sqrt(c)*exp(log((sqrt(4 - 2*c) + 2)/(2 - sqrt(4 - 2*c)))/sqrt(4 - 2*c))/2
limit c->0+ r_min/M = 2
ROUTE_D_RMIN_C0_LIMIT_PASS = True


# REP18.18 — Classification des quatre routes

### Route A
\[
c_1+c_4=0
\]

- obstruction Schwarzschild connue annulée ;
- mais \(K_V=0\) ;
- branche vectorielle générique perdue ;
- **nouvel audit canonique nécessaire**.

### Route B
\[
c_1+c_2+c_3=0
\]

- obstruction chute libre annulée ;
- mais la contrainte scalaire change de rang et impose \(\dot\psi=0\) sous stabilité tensorielle ;
- **nouvel audit canonique nécessaire**.

### Route C
\[
\eta_\infty\neq0
\]

- génériquement densité d'action radiale non décroissante ;
- pas de réparation asymptotique générique propre ;
- surface spéciale \(c_1+2c_2+c_3=0\) non classifiée.

### Route D
\[
\text{métrique sphérique déformée}
\]

- ne force ni \(c_1+c_4=0\) ni \(c_1+c_2+c_3=0\) ;
- conserve donc la possibilité de rester sur la branche faible champ générique stable ;
- fournit un système exact réduit et une solution par quadrature ;
- reproduit le potentiel newtonien au premier ordre ;
- **ne constitue pas un PASS Schwarzschild exact**.

La route D est la seule route qui produit ce tour un candidat de réparation classique concret sans quitter immédiatement la branche héritée.

In [11]:
# REP18.19 — Final classifier
ROUTE_A_DIRECT_CHAIN_PRESERVING_REPAIR = False
ROUTE_B_DIRECT_CHAIN_PRESERVING_REPAIR = False
ROUTE_C_DIRECT_CHAIN_PRESERVING_REPAIR = False

ROUTE_D_DEFORMED_SPHERICAL_CANDIDATE_MATERIALIZED = all([
    ROUTE_D_REDUCED_EOMS_MATERIALIZED,
    ROUTE_D_A_EQUATION_CROSSCHECK_PASS,
    ROUTE_D_Y_REDUCTION_CROSSCHECK_PASS,
    ROUTE_D_QUADRATURE_IDENTITIES_PASS,
    ROUTE_D_ASYMPTOTIC_SERIES_PASS,
    ROUTE_D_C0_RECOVERS_SCHWARZSCHILD,
    ROUTE_D_RMIN_C0_LIMIT_PASS,
])

ROUTE_D_PRESERVES_GENERIC_COUPLING_BRANCH_POSSIBILITY = True
ROUTE_D_PRESERVES_NEWTONIAN_LEADING_TERM = True
ROUTE_D_FULL_COVARIANT_FIELD_EQUATIONS_VERIFIED = False
ROUTE_D_INNER_REGULARITY_VERIFIED = False
ROUTE_D_PPN_OBSERVATIONAL_VIABILITY_VERIFIED = False

CLASSICAL_CORE_REPAIR_CANDIDATE_FOUND = (
    ROUTE_D_DEFORMED_SPHERICAL_CANDIDATE_MATERIALIZED
)

EXACT_SCHWARZSCHILD_BENCHMARK_PASS = False
SCHWARZSCHILD_BENCHMARK_PASS = False
SCHWARZSCHILD_BENCHMARK_AUTHORIZED = False

DEFORMED_SPHERICAL_GVH_BENCHMARK_AUTHORIZED = (
    CLASSICAL_CORE_REPAIR_CANDIDATE_FOUND
)

KERR_BENCHMARK_AUTHORIZED = False
REAL_DATA_PARAMETER_INFERENCE_AUTHORIZED = False
QUANTIZATION_READY = False

REP18_OBSTRUCTIONS = [
    "DEFORMED-SPHERICAL-CANDIDATE-FULL-COVARIANT-EOM-NOT-YET-CROSSCHECKED",
    "DEFORMED-SPHERICAL-INNER-REGULARITY-NOT-YET-CLASSIFIED",
    "PPN-AND-OBSERVATIONAL-VIABILITY-NOT-YET-CLASSIFIED",
    "DEGENERATE-ROUTES-A-B-REQUIRE-INDEPENDENT-CONSTRAINT-AUDITS",
]

REP18_LOCAL_AUDIT_PASS = all([
    UPSTREAM_GATE,
    ROUTE_A_VECTOR_HESSIAN_ZERO,
    ROUTE_B_NON_DIVIDED_CONSTRAINT_PASS,
    ROUTE_C_RADIAL_DENSITY_CONSTANT,
    ROUTE_D_DEFORMED_SPHERICAL_CANDIDATE_MATERIALIZED,
    CLASSICAL_CORE_REPAIR_CANDIDATE_FOUND,
    not EXACT_SCHWARZSCHILD_BENCHMARK_PASS,
    not KERR_BENCHMARK_AUTHORIZED,
    not REAL_DATA_PARAMETER_INFERENCE_AUTHORIZED,
    not QUANTIZATION_READY,
])

REP18_NEXT_AUTHORIZED = (
    "AUDIT-DEFORMED-SPHERICAL-GVH-VACUUM-FULL-EQUATIONS-INNER-REGULARITY-AND-PPN"
    if REP18_LOCAL_AUDIT_PASS
    else "REPAIR-.3.3.18-ESCAPE-ROUTES-AUDIT"
)

assert REP18_LOCAL_AUDIT_PASS

print("ROUTE_A_DIRECT_CHAIN_PRESERVING_REPAIR =", ROUTE_A_DIRECT_CHAIN_PRESERVING_REPAIR)
print("ROUTE_B_DIRECT_CHAIN_PRESERVING_REPAIR =", ROUTE_B_DIRECT_CHAIN_PRESERVING_REPAIR)
print("ROUTE_C_DIRECT_CHAIN_PRESERVING_REPAIR =", ROUTE_C_DIRECT_CHAIN_PRESERVING_REPAIR)
print("ROUTE_D_DEFORMED_SPHERICAL_CANDIDATE_MATERIALIZED =", ROUTE_D_DEFORMED_SPHERICAL_CANDIDATE_MATERIALIZED)
print("CLASSICAL_CORE_REPAIR_CANDIDATE_FOUND =", CLASSICAL_CORE_REPAIR_CANDIDATE_FOUND)
print("EXACT_SCHWARZSCHILD_BENCHMARK_PASS =", EXACT_SCHWARZSCHILD_BENCHMARK_PASS)
print("DEFORMED_SPHERICAL_GVH_BENCHMARK_AUTHORIZED =", DEFORMED_SPHERICAL_GVH_BENCHMARK_AUTHORIZED)
print("KERR_BENCHMARK_AUTHORIZED =", KERR_BENCHMARK_AUTHORIZED)
print("REAL_DATA_PARAMETER_INFERENCE_AUTHORIZED =", REAL_DATA_PARAMETER_INFERENCE_AUTHORIZED)
print("REP18_OBSTRUCTIONS =", REP18_OBSTRUCTIONS)
print("REP18_NEXT_AUTHORIZED =", REP18_NEXT_AUTHORIZED)

ROUTE_A_DIRECT_CHAIN_PRESERVING_REPAIR = False
ROUTE_B_DIRECT_CHAIN_PRESERVING_REPAIR = False
ROUTE_C_DIRECT_CHAIN_PRESERVING_REPAIR = False
ROUTE_D_DEFORMED_SPHERICAL_CANDIDATE_MATERIALIZED = True
CLASSICAL_CORE_REPAIR_CANDIDATE_FOUND = True
EXACT_SCHWARZSCHILD_BENCHMARK_PASS = False
DEFORMED_SPHERICAL_GVH_BENCHMARK_AUTHORIZED = True
KERR_BENCHMARK_AUTHORIZED = False
REAL_DATA_PARAMETER_INFERENCE_AUTHORIZED = False
REP18_OBSTRUCTIONS = ['DEFORMED-SPHERICAL-CANDIDATE-FULL-COVARIANT-EOM-NOT-YET-CROSSCHECKED', 'DEFORMED-SPHERICAL-INNER-REGULARITY-NOT-YET-CLASSIFIED', 'PPN-AND-OBSERVATIONAL-VIABILITY-NOT-YET-CLASSIFIED', 'DEGENERATE-ROUTES-A-B-REQUIRE-INDEPENDENT-CONSTRAINT-AUDITS']
REP18_NEXT_AUTHORIZED = AUDIT-DEFORMED-SPHERICAL-GVH-VACUUM-FULL-EQUATIONS-INNER-REGULARITY-AND-PPN


# REP18.20 — Portée scientifique

`.3.3.18` ne répare pas rétroactivement Schwarzschild exact.

Le résultat central est différent :

\[
\boxed{
\text{la route D produit une solution sphérique GVH déformée candidate}
}
\]

sur la même région de couplages où :

\[
K_V=c_1+c_4>0,
\]

et :

\[
G_S=\frac{2(2-c_1-c_4)}{c_1+c_4}>0.
\]

Elle satisfait asymptotiquement :

\[
N^2
=
1-\frac{2M}{r}
-\frac{c_1+c_4}{6}
\left(\frac Mr\right)^3+\cdots,
\]

et :

\[
A^2
=
1+\frac{2M}{r}
+
\left[
4+\frac{c_1+c_4}{2}
\right]
\left(\frac Mr\right)^2
+\cdots.
\]

Cela ouvre une nouvelle question falsifiable :

\[
\boxed{
\text{les corrections contrôlées par }c_1+c_4
\text{ sont-elles compatibles avec les tests gravitationnels ?}
}
\]

Mais avant les données, il faut encore :

1. re-vérifier les EOM covariantes complètes ;
2. classifier \(r_{\min}\) : throat, bord régulier ou singularité ;
3. dériver les observables PPN/fort champ ;
4. seulement ensuite confronter aux données.

In [12]:
# REP18.21 — Machine-readable artifact
artifact = {
    "notebook": "GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.18_Classical_Core_Repair_and_Schwarzschild_Escape_Routes_Audit_FAST",
    "execution_scope": "CLASSICAL_CORE_REPAIR_AND_SCHWARZSCHILD_ESCAPE_ROUTES",
    "upstream": UPSTREAM,
    "route_A": {
        "condition": "c1+c4=0",
        "vector_hessian_zero": ROUTE_A_VECTOR_HESSIAN_ZERO,
        "generic_vector_pass_transferable": ROUTE_A_GENERIC_VECTOR_PASS_TRANSFERABLE,
        "requires_fresh_constraint_dof_audit": ROUTE_A_REQUIRES_FRESH_CONSTRAINT_DOF_AUDIT,
    },
    "route_B": {
        "condition": "c1+c2+c3=0",
        "non_divided_scalar_constraint": str(dq_B),
        "under_tensor_stability": "dot(psi)=0",
        "generic_scalar_pass_transferable": ROUTE_B_GENERIC_SCALAR_PASS_TRANSFERABLE,
        "requires_fresh_constraint_dof_audit": ROUTE_B_REQUIRES_FRESH_CONSTRAINT_DOF_AUDIT,
    },
    "route_C": {
        "boundary": "eta -> eta_inf != 0 finite",
        "radial_action_density_asymptotic": str(Lrad_inf),
        "generic_finite_action_compatible": ROUTE_C_GENERIC_FINITE_ACTION_COMPATIBLE,
        "special_surface": ROUTE_C_SPECIAL_CODIM1_SURFACE,
    },
    "route_D": {
        "candidate": "aligned unit GVH vector on deformed static spherical metric",
        "c": "c1+c4",
        "P_y": str(P),
        "equations": {
            "A_squared": "P(y)=1+2y+(c/2)y^2",
            "y_ode": "r y'=-y P(y)",
            "dlogN_dy": "-1/P(y)",
        },
        "asymptotic": {
            "N_squared": "1-2x-(c/6)x^3+O(x^4), x=M/r",
            "A_squared": "1+2x+(4+c/2)x^2+(8+5c/2)x^3+O(x^4)",
            "newtonian_leading_term_preserved": ROUTE_D_PRESERVES_NEWTONIAN_LEADING_TERM,
        },
        "GR_limit": {
            "c_zero_recovers_exact_Schwarzschild": ROUTE_D_C0_RECOVERS_SCHWARZSCHILD,
        },
        "inner_limit": {
            "domain": ROUTE_D_INHERITED_C_DOMAIN,
            "I_inf": str(Iinf),
            "N_min": str(Nmin),
            "r_min_over_M": str(rmin_over_M),
            "r_min_over_M_c_to_0": str(rmin_limit_c0),
            "full_regularity_verified": ROUTE_D_INNER_REGULARITY_VERIFIED,
        },
        "full_covariant_EOM_verified": ROUTE_D_FULL_COVARIANT_FIELD_EQUATIONS_VERIFIED,
        "PPN_observational_viability_verified": ROUTE_D_PPN_OBSERVATIONAL_VIABILITY_VERIFIED,
    },
    "scientific_status": {
        "CLASSICAL_CORE_REPAIR_CANDIDATE_FOUND": CLASSICAL_CORE_REPAIR_CANDIDATE_FOUND,
        "EXACT_SCHWARZSCHILD_BENCHMARK_PASS": EXACT_SCHWARZSCHILD_BENCHMARK_PASS,
        "SCHWARZSCHILD_BENCHMARK_PASS": SCHWARZSCHILD_BENCHMARK_PASS,
        "SCHWARZSCHILD_BENCHMARK_AUTHORIZED": SCHWARZSCHILD_BENCHMARK_AUTHORIZED,
        "DEFORMED_SPHERICAL_GVH_BENCHMARK_AUTHORIZED": DEFORMED_SPHERICAL_GVH_BENCHMARK_AUTHORIZED,
        "KERR_BENCHMARK_AUTHORIZED": KERR_BENCHMARK_AUTHORIZED,
        "REAL_DATA_PARAMETER_INFERENCE_AUTHORIZED": REAL_DATA_PARAMETER_INFERENCE_AUTHORIZED,
        "QUANTIZATION_READY": QUANTIZATION_READY,
    },
    "verdict": {
        "REP18_LOCAL_AUDIT_PASS": REP18_LOCAL_AUDIT_PASS,
        "obstructions": REP18_OBSTRUCTIONS,
    },
    "next_authorized": REP18_NEXT_AUTHORIZED,
    "scope_note": "Routes A/B are degenerate branches requiring fresh constraint analyses; C is not a generic finite-action asymptotic repair; D supplies a deformed spherical candidate preserving the possibility of the inherited generic stable weak-field branch. Exact Schwarzschild remains failed."
}

export_dir = Path("/content/gvh_exports") if Path("/content").exists() else Path("/mnt/data")
export_dir.mkdir(parents=True, exist_ok=True)
artifact_path = export_dir / "gvh_0.3.2.7.3.7.3.3.18_Classical_Core_Repair_and_Schwarzschild_Escape_Routes_Audit_FAST.json"
artifact_path.write_text(json.dumps(artifact, indent=2, ensure_ascii=False), encoding="utf-8")
print("REP18 artifact =", artifact_path)

REP18 artifact = /content/gvh_exports/gvh_0.3.2.7.3.7.3.3.18_Classical_Core_Repair_and_Schwarzschild_Escape_Routes_Audit_FAST.json


# Conclusion

Les routes A et B ne sont pas des réparations gratuites : elles changent le rang du système linéarisé.

La route C n'est pas génériquement compatible avec la classe asymptotique finie retenue.

La route D est la seule qui fournisse, dans ce tour, un candidat classique concret sans imposer la sortie immédiate de la branche faible champ générique stable :

\[
\boxed{
A^2=1+2y+\frac{c_1+c_4}{2}y^2,
\qquad
ry'=-yA^2.
}
\]

Mais :

\[
\boxed{
\texttt{EXACT\_SCHWARZSCHILD\_BENCHMARK\_PASS=False}.
}
\]

La prochaine marche autorisée est un audit complet du **candidat sphérique GVH déformé**, pas Kerr et pas encore les données réelles.